In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [4]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
print(df.shape)
df.head()

(100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [5]:
print('Duplicate rows:', df.duplicated().sum())
print('Target balance:')
print(df['diabetes'].value_counts())
print(df['diabetes'].value_counts(normalize=True)*100)
print('gender:', df['gender'].unique())
print('smoking_history:', df['smoking_history'].unique())

Duplicate rows: 3854
Target balance:
diabetes
0    91500
1     8500
Name: count, dtype: int64
diabetes
0    91.5
1     8.5
Name: proportion, dtype: float64
gender: <StringArray>
['Female', 'Male', 'Other']
Length: 3, dtype: str
smoking_history: <StringArray>
['never', 'No Info', 'current', 'former', 'ever', 'not current']
Length: 6, dtype: str


In [6]:
df = df.drop_duplicates()
print(df.shape)

(96146, 9)


In [7]:
df_encoded = pd.get_dummies(df, columns=['gender'],dtype=int)
df_encoded = df_encoded.drop(columns=['gender_Other'])
df_encoded = pd.get_dummies(df_encoded, columns=['smoking_history'],drop_first=True,dtype=int)
df_encoded.head()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes,gender_Female,gender_Male,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
0,80.0,0,1,25.19,6.6,140,0,1,0,0,0,0,1,0
1,54.0,0,0,27.32,6.6,80,0,1,0,0,0,0,0,0
2,28.0,0,0,27.32,5.7,158,0,0,1,0,0,0,1,0
3,36.0,0,0,23.45,5.0,155,0,1,0,1,0,0,0,0
4,76.0,1,1,20.14,4.8,155,0,0,1,1,0,0,0,0


In [8]:
x = df_encoded.drop(columns='diabetes')
y = df_encoded['diabetes']

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=2
)

numeric_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

scaler = StandardScaler()
scaler.fit(x_train_raw[numeric_cols])

x_train = x_train_raw.copy()
x_test = x_test_raw.copy()
x_train[numeric_cols] = scaler.transform(x_train_raw[numeric_cols])
x_test[numeric_cols] = scaler.transform(x_test_raw[numeric_cols])

print(x_train.shape, x_test.shape)

(76916, 13) (19230, 13)


In [13]:
logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(x_train, y_train)
# print(pd.Series(logreg.coef_[0], index=x_train.columns))
# print("Bias:", logreg.intercept_)
print('Model Trained')

Model Trained


In [10]:
print('=== Logistic Regression ===')
pred_lr = logreg.predict(x_test)
print(classification_report(y_test, pred_lr))
print('ROC-AUC:', roc_auc_score(y_test, logreg.predict_proba(x_test)[:, 1]))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred_lr))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.99      0.89      0.93     17534
           1       0.43      0.88      0.58      1696

    accuracy                           0.89     19230
   macro avg       0.71      0.88      0.76     19230
weighted avg       0.94      0.89      0.90     19230

ROC-AUC: 0.9612315378908041
Confusion matrix:
[[15548  1986]
 [  202  1494]]


In [11]:
def predict_diabetes(age, gender, hypertension, heart_disease, smoking_history, bmi, hba1c, glucose, model=logreg):
    row = pd.DataFrame([{
        'age': age, 'hypertension': hypertension, 'heart_disease': heart_disease,
        'bmi': bmi, 'HbA1c_level': hba1c, 'blood_glucose_level': glucose,
        'gender': gender, 'smoking_history': smoking_history
    }])
    row_encoded = pd.get_dummies(row, columns=['gender', 'smoking_history'])
    # add any one-hot columns missing from this single row, in the training column order
    row_encoded = row_encoded.reindex(columns=x.columns, fill_value=0)
    row_encoded[numeric_cols] = scaler.transform(row_encoded[numeric_cols])
    pred = model.predict(row_encoded)[0]
    return 'Diabetic' if pred == 1 else 'Not diabetic'

print(predict_diabetes(age=44, gender='Female', hypertension=0, heart_disease=0,
                        smoking_history='never', bmi=19.31, hba1c=6.5, glucose=200))

Diabetic
